# Flare Finder Pipeline — HST-COS FUV Analysis
### Young M Dwarf Flare (YMDF) Framework

This notebook implements the **Flare Finder** pipeline from the YMDF framework for the detection, characterisation, and recovery of stellar flares in HST-COS far-ultraviolet (FUV) time-series data, Mamonova et al 2025 doi:[10.1051/0004-6361/202554614]

The pipeline follows these steps:
1. **Load** the stellar sample catalogue and raw light curve data
2. **Detrend** the light curve using a Savitzky–Golay filter
3. **Detect** flares using sigma-clipping and multi-threshold criteria
4. **Recover** flare completeness via injection-recovery simulations
5. **Characterise** flares with corrected equivalent durations and energies
6. **Save** the final flare catalogue

This notebook is configured for: HST-COS 10-second cadence FUV data, all parameters are applicable for FUV data.

In [ ]:
## 1. Imports
import pandas
import numpy
import matplotlib.pyplot as plt
from astropy import units, constants
from astropy.table import Table
from typing import Tuple, List, Optional, Any
import seaborn
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pandas.options.mode.chained_assignment = None

from phab.ymdf.flare_finder import (
    findFlares,
    findGaps,
    equivalentDurationModel,
    sigmaClip,
    addPaddingToList,
    detrendSavGol,
    luminosityQuiescent,
    fluxQuiescent,
    findIterativeMedian,
    sampleFlareRecovery,
    characterizeFlares
)

## 2. Stellar Sample Catalogue

Load the pre-compiled stellar sample catalogue (`pandas` DataFrame) containing stellar parameters (ID, radius, effective temperature, distance, etc.) for all targets in the young moving group sample. The catalogue can be stored as a `pandas` pickle file/csv.

In [ ]:
enriched = pandas.read_pickle("./Newsample_reconfirmed.pkl")
print(f"Sample size: {len(enriched)} stars")
print(enriched.head())

## 3. Target Configuration

In [ ]:
# convention for the observation files' names
obs    = 1
spur   = 1
prefix = [""]
star   = "2MASS J08440915-7833457"
sector = f"cos{obs}{prefix[0]}"

# Stellar parameters pulled from the sample catalogue
R_star    = enriched.at[star, "star_radius"]
star_teff = enriched.at[star, "teff_lit"]
star_dist = enriched.at[star, "Distance"]

print(f"Target:    {star}")
print(f"Sector:    {sector}")
print(f"R_star:    {R_star} R_sun")
print(f"T_eff:     {star_teff} K")
print(f"Distance:  {star_dist} pc")

## 4. Loading the HST-COS Light Curve

Load the HST-COS FUV time-series from a CSV file produced by the data reduction pipeline CALCOS. The file contains three columns: time (seconds), flux, and flux error. Flux and error are cast to `float32` to reduce memory usage. The raw time, flux, and error arrays are also extracted as NumPy arrays for use in downstream functions.

In [ ]:
data12 = pandas.read_csv(
    f"./MAST_{star}_observation_{obs}_10SEC/HST/MAST_{star}_observation_{obs}_tot_flux_10s.csv", # the file name is a convention
    names=["time", "flux", "error"],
    skiprows=1
)

data12["flux"]  = data12["flux"].astype("float32")
data12["error"] = data12["error"].astype("float32")
data12["fluxError"] = data12["error"].values

timeSeries       = data12["time"].values
fluxSeries       = data12["flux"].values
fluxErrorSeries  = data12["error"].values

print(f"Light curve loaded: {len(data12)} data points")
print(f"Time range: {timeSeries[0]:.1f} – {timeSeries[-1]:.1f} s")
print(f"Median flux: {numpy.nanmedian(fluxSeries):.4f}")


## 5. Gap Detection and Detrending

HST-COS observations are interrupted by Earth occultations, producing regular gaps in the time series. The `findGaps` function identifies these gaps using a maximum allowed gap of 30 seconds and a minimum continuous observation period of 10 seconds. The `detrendSavGol` function then fits and removes the slowly varying quiescent stellar continuum within each continuous segment using a Savitzky–Golay filter, producing a detrended light curve suitable for flare detection.

In [ ]:
gaps = findGaps(
    data12,
    maximumGap=30,
    minimumObservationPeriod=10
)
print(f"Number of continuous segments: {len(gaps)}")

data1 = detrendSavGol(
    data12,
    gaps,
    padding=3,
    waveRange="uv",
    windowLength=0
)
data1 = data1.dropna(subset=["time"])

fig1, ax = plt.subplots(figsize=(15, 4))
ax.plot(
    data12["time"],
    data12["flux"] / numpy.nanmedian(data12["flux"]) + 0.001,
    label="Raw flux"
)
ax.plot(
    data1["time"],
    data1["fluxDetrended"] / numpy.nanmedian(data1["fluxDetrended"]),
    label="Detrended flux"
)
ax.legend()
fig1.supxlabel("Time [s]", fontsize=14)
plt.show()

## 6. Flare Detection

Flares are identified in the detrended light curve using a multi-threshold sigma-clipping algorithm implemented in `findFlares`.  A candidate flare requires the detrended flux to exceed the local noise level by at least `n1=3` sigma, and to simultaneously exceed `n2=2` times the combined sigma and detrended flux error. At least `n3=3` consecutive points must satisfy these criteria for an event to be flagged as a flare. A minimum separation of 3 data points is enforced between distinct flare events. The function returns the updated detrended light curve `data1` and a flare table `fla1` containing the detected event: start and stop in indices and actual time, recovered equivalent duration, recovered amplitude, duration and the total valid data points.


In [ ]:
data1, fla1 = findFlares(
    data12,
    n1=3,
    n2=2,
    n3=3,
    minSep=3,
    padding=3,
    maximumGap=30,
    minimumObservationPeriod=10,
    waveRange="uv"
)

print(f"Flares detected: {len(fla1)}")
print(fla1)
print(f"Detrended flux range: {data1['fluxDetrended'].min():.4f} – {data1['fluxDetrended'].max():.4f}")

## 8. Injection-Recovery Simulations

To characterise the completeness of the flare detection as a function of flare amplitude and duration, synthetic flares are injected into the observed light curve and the recovery rate is measured. The `sampleFlareRecovery` function generates 3500 synthetic flares drawn from a uniform distribution in amplitude (0.01–200 normalised flux units) and duration (10–10000 s), injects them one at a time into the detrended light curve, and attempts to recover each using the same detection thresholds as in the previous step. The injection rate is set to one synthetic flare per observation second (`fakefreq=1./24/60/60`), with a maximum of one injected flare per continuous segment to avoid confusion between overlapping events.

The output consists of two tables:
- `fla_rec` — the recovered flares
- `fla_inj` — the properties of all injected synthetic flares

In [ ]:
fla_rec, fla_inj = sampleFlareRecovery(
    data1,
    fla1,
    iterations=3500,
    n1=3,
    n2=2,
    n3=3,
    minSep=3,
    padding=3,
    maximumGap=30,
    minimumObservationPeriod=10,
    ampl=[0.01, 200.],
    dur=[10., 10000.],
    fakefreq=1./24/60/60,
    maxFlaresPerGap=1
)

print(f"Total injected flares: {len(fla_inj)}")
print(f"Total recovered flares: {len(fla_rec)}")

## 9. Flare Characterisation

The `characterizeFlares` function uses the injection-recovery results to compute completeness-corrected equivalent durations (`ed_corr`) for each detected flare. 2D recovery heatmaps in amplitude–duration space for probability and ED ratio is generated to visualise the detection completeness across the parameter space sampled by the injection-recovery simulations. The corrected equivalent durations can subsequently be converted to flare energies using the quiescent stellar luminosity.

In [ ]:
fla3 = characterizeFlares(
    fla_rec,
    fla_inj,
    ampl=[0.01, 200.],
    dur=[10., 10000.],
    plot_heatmap=True
)

print(f"Characterised flares: {len(fla3)}")
print(fla3)

## 10. Saving Results

The final characterised flare catalogue is saved in both CSV and pickle.

In [ ]:
fla3.to_csv(f"./new_sample_flares/{star}_{sector}_flares_recovered.csv")
fla3.to_pickle(f"./new_sample_flares/{star}_{sector}_flares_recovered.pkl")

print(f"Saved: ./new_sample_flares/{star}_{sector}_flares_recovered.csv")
print(f"Saved: ./new_sample_flares/{star}_{sector}_flares_recovered.pkl")